google collab depedencies

In [1]:
!pip -q install bertopic
!pip -q install sastrawi
!pip -q install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 6.2 MB/s eta 0:00:00


In [2]:
!git clone -q -b gavriel-thesis https://github.com/ranslemus/topic_modeling_KBMI4.git
%cd topic_modeling_KBMI4

/kaggle/working/topic_modeling_KBMI4


In [3]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px
import random

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from hdbscan.validity import validity_index

# for linux
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

# for windows
# import umap as UMAP
# import hdbscan as HDBSCAN

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : Tesla T4


In [5]:
df = pd.read_csv("data/preprocessed_data_downsampled.csv")
df = df[df["year"]==2025]
df.head()

,reviewId,bank,score,year,text
3,629f06db-dc19-4a6b-a526-c5fa09933ed2,LIVIN_MANDIRI_REVIEWS,2,2025,kenapa di login tidak bisa ya malah muncul tul...
10,33535e95-15cb-49b3-bcb7-894957cb159d,WONDR_BNI_REVIEWS,1,2025,ngelag mulu deh
12,8091018d-801d-4a9f-a3ff-86491d781b1e,BRIMO_REVIEWS,1,2025,transaksi berhasil uang enggak masuk gimnaa si...
13,658c217f-74b2-4280-b07a-7ab529fd97a1,BCAMOBILE_REVIEWS,2,2025,sering keluar harus verifikasi lagi terus luma...
17,47ed6779-21fd-45bb-a5a6-c6d92ef93186,BCAMOBILE_REVIEWS,1,2025,malu ih bca mah


In [6]:
df["word_count"] = df["text"].astype(str).str.split().apply(len)
df = df[df["word_count"] >= 5].reset_index(drop=True)
print(f"Total documents setelah filter: {len(df):,}")

Total documents setelah filter: 42,661


In [7]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 42,661


# IndoBERT

In [8]:
MODEL_NAME = "LazarusNLP/simcse-indobert-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)

model.eval()

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: LazarusNLP/simcse-indobert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(50000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [9]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

In [10]:
def encode_documents(
    documents,
    batch_size=32,
    max_length=128
):

    embeddings = []

    with torch.no_grad():

        for i in tqdm(
            range(0, len(documents), batch_size)
        ):

            batch = documents[
                i:i+batch_size
            ]

            encoded_input = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded_input = {
                k: v.to(device)
                for k, v in encoded_input.items()
            }

            model_output = model(**encoded_input)

            sentence_embeddings = mean_pooling(
                model_output,
                encoded_input["attention_mask"]
            )

            sentence_embeddings = (
                sentence_embeddings
                .cpu()
                .numpy()
            )

            embeddings.append(sentence_embeddings)

    return np.vstack(embeddings)

In [11]:
embeddings = encode_documents(
    documents,
    batch_size=32,
    max_length=128
)

  0%|          | 0/1334 [00:00<?, ?it/s]

In [12]:
print(embeddings.shape)

(42661, 768)


In [13]:
embeddings[0]

array([ 1.31894362e+00,  9.60366786e-01,  1.30218878e-01, -6.16801903e-02,
        4.10509288e-01, -7.32329428e-01, -1.66571188e+00,  8.28113317e-01,
        8.47559512e-01,  4.62959141e-01, -1.03036702e+00, -1.15735888e+00,
       -9.49390411e-01, -1.97022617e-01, -4.13674146e-01, -5.59255257e-02,
       -8.73682320e-01, -3.98348600e-01,  9.12975729e-01,  1.10419989e-01,
        1.62639177e+00,  6.93946123e-01,  2.03165129e-01, -3.73411298e-01,
       -7.23186851e-01, -1.41948688e+00,  2.93903202e-01, -4.36818421e-01,
       -4.34622020e-01, -1.46654904e-01,  3.04300129e-01,  3.05604726e-01,
        1.74515486e+00,  6.05405420e-02,  2.28680789e-01, -9.99139808e-03,
        6.52468562e-01,  1.10717797e+00, -1.17894602e+00,  1.68272883e-01,
        4.44559038e-01, -1.43568218e-01,  7.49856293e-01, -8.61199439e-01,
       -4.08587456e-01,  2.14764491e-01,  1.07191110e+00,  1.40870261e+00,
        1.30559671e+00,  1.03162718e+00, -1.46915901e+00, -1.55424818e-01,
       -5.29873371e-01,  

In [14]:
norms = np.linalg.norm(embeddings, axis=1)

print("Minimum Norm :", norms.min())
print("Maximum Norm :", norms.max())
print("Average Norm :", norms.mean())
print("Std Norm :", norms.std())

Minimum Norm : 15.018309
Maximum Norm : 26.08848
Average Norm : 21.370525
Std Norm : 1.9197338


In [15]:
print("NaN :", np.isnan(embeddings).sum())
print("Inf :", np.isinf(embeddings).sum())

NaN : 0
Inf : 0


In [16]:
# np.save(
#     "embeddings/indobert_embeddings_downsampled_cutted.npy",
#     embeddings
# )

# BERTopic

In [41]:
# embeddings = np.load("embeddings/indobert_embeddings_downsampled_full.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (42661, 768)


In [42]:
sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

# extra_particles = ["banget", "terus", "padahal", "sih", "aja", "saja", "dong", "deh", "ya", "kok", "biar", "gitu", "nih", "loh", "mau", "sudah", "belum"]
sastrawi_stopwords_extended = sastrawi_stopwords 

vectorizer_model = CountVectorizer(
  ngram_range=(1,2),
  stop_words=sastrawi_stopwords,
  token_pattern=r"(?u)\b[^\d\W]+\b",
  min_df=2,
  )

baseline UMAP for testing purpose

In [43]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [44]:
hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="leaf",
    prediction_data=True
)

In [45]:
from bertopic.vectorizers import ClassTfidfTransformer

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [46]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

In [47]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-11 06:50:08,501 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-11 06:50:10,224 - BERTopic - Dimensionality - Completed ✓
2026-08-11 06:50:10,227 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-11 06:50:10,565 - BERTopic - Cluster - Completed ✓
2026-08-11 06:50:10,576 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-11 06:50:12,036 - BERTopic - Representation - Completed ✓


# Evaluation

Basic Statistics

In [48]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,29478,-1_bri_begini_uang_bayar,"[bri, begini, uang, bayar, kayak, suruh, dong,...",[astaga ribet banget baru pertama kali instal ...
1,0,384,0_verifikasi wajah_wajah_wajah gagal_wajah susah,"[verifikasi wajah, wajah, wajah gagal, wajah s...",[aplikasi enggak jelas verifikasi wajah saja s...
2,1,381,1_bni mobile_mobile_mobile banking_bni,"[bni mobile, mobile, mobile banking, bni, mend...",[sejak pakai wonder sering error enakan mobile...
3,2,375,2_pemeliharaan_maintenance_tiap malam_malam,"[pemeliharaan, maintenance, tiap malam, malam,...",[transaksi gagal saldo enggak balik tiap tenga...
4,3,330,3_meminta update_update terus_update mulu_seri...,"[meminta update, update terus, update mulu, se...",[update mulu 2 hari sekali meminta update teru...
5,4,329,4_nama_scroll_pencarian_search,"[nama, scroll, pencarian, search, manual, cari...",[setelah update sekarang mau transfer harus ca...
6,5,328,5_foto ktp_foto_ktp_upload,"[foto ktp, foto, ktp, upload, upload foto, ktp...","[susah sekali upload foto ktp, selalu upload f..."
7,6,309,6_otp_kode otp_otp enggak_kode,"[otp, kode otp, otp enggak, kode, otp nya, men...",[waktu untuk otp 1 menit sampai 3 kali meminta...
8,7,288,7_uninstall_uninstal_hapus_download,"[uninstall, uninstal, hapus, download, downloa...",[bagus tapi ribet saya kira cuma saya yang men...
9,8,259,8_tunai_tarik tunai_tarik_setor,"[tunai, tarik tunai, tarik, setor, setor tunai...","[fitur tarik tunai tanpa kartu belum ada ya, f..."


In [49]:
num_topics = len(topic_info) - 1

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 138
Outliers            : 29,478
Outlier Percentage  : 69.10%


Topic Size

In [50]:
topic_info[["Topic","Count"]]

,Topic,Count
0,-1,29478
1,0,384
2,1,381
3,2,375
4,3,330
...,...,...
134,133,32
135,134,32
136,135,31
137,136,31


Top Words

In [51]:
topic_summary = []
for topic in topic_info.Topic:
    if topic == -1:
        continue
    words = ", ".join([w for w, _ in topic_model.get_topic(topic)[:10]])
    topic_summary.append({"Topic": topic, "Count": topic_info.loc[topic_info.Topic==topic, "Count"].values[0], "Top Words": words})

topic_summary_df = pd.DataFrame(topic_summary).sort_values("Count", ascending=False)
topic_summary_df

,Topic,Count,Top Words
0,0,384,"verifikasi wajah, wajah, wajah gagal, wajah su..."
1,1,381,"bni mobile, mobile, mobile banking, bni, mendi..."
2,2,375,"pemeliharaan, maintenance, tiap malam, malam, ..."
3,3,330,"meminta update, update terus, update mulu, ser..."
4,4,329,"nama, scroll, pencarian, search, manual, cari,..."
...,...,...,...
133,133,32,"tf dana, enggak tf, sekarang tf, tf sesama, tf..."
134,134,32,"penguna brimo, lemott, penguna, tf kalau, bri ..."
135,135,31,"video call, video, vidio, vidio call, call, ca..."
136,136,31,"persulit, daftar persulit, persulit mau, login..."


Representative Reviews

In [52]:
TOP_N_TOPICS_TO_INSPECT = 15  # cukup buat cek kualitas, nggak perlu semua 30-62 topik

top_topics = topic_info[topic_info.Topic != -1].nlargest(TOP_N_TOPICS_TO_INSPECT, "Count")["Topic"].tolist()
representative_docs = topic_model.get_representative_docs()

for topic in top_topics:
    print(f"\n{'='*80}\nTOPIC {topic} (n={topic_info.loc[topic_info.Topic==topic,'Count'].values[0]})")
    for i, doc in enumerate(representative_docs[topic][:3], 1):
        print(f"{i}. {doc}")


TOPIC 0 (n=384)
1. aplikasi enggak jelas verifikasi wajah saja susah banget gagal mulu
2. buat verifikasi wajah susah banget
3. kenapa enggak bisa verifikasi wajah dan selalu gagal ya

TOPIC 1 (n=381)
1. sejak pakai wonder sering error enakan mobile banking yang lama wonder kalo malam suka gangguan kadang saldo tiba-tiba ngurangin harus menunggu beberapa jam muncul lagi saldo normalnya parah harusnya kalo belum siap apk wonder jangan dulu diresmikan mending stay dulu di mobile banking
2. bni mobile banking sudah tidak ada kah
3. lebih bagus pakai bni mobile banking yang lama bni mobile banking yang lama lebih simpel dan praktis

TOPIC 2 (n=375)
1. transaksi gagal saldo enggak balik tiap tengah malam enggak bisa trx apapun bhkan skdr cek saldo ribet banget enggak jelas sampai jam berapa maintenance nya mnaa tiap hari minimali tiap ada pemeliharaan rutin di notifkan ke tiap pengguna jadi bisa ngindari trx di jam2 pemeliharaan
2. kebiasaan banget kalo pemeliharaan jam malam semua juga ad

silhoutte score

In [53]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.5483


In [54]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info.Topic:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]
    topic_words.append(words)

unique_words = len(
    set(chain.from_iterable(topic_words))
)

total_words = len(topic_words) * top_n
topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.8906


Representative Reviews

In [55]:
random.seed(42)

sample_size = 20

for topic_id in sorted(set(topics)):

    if topic_id == -1:
        continue

    topic_docs = [
        doc for doc, topic in zip(documents, topics)
        if topic == topic_id
    ]

    n = min(sample_size, len(topic_docs))
    sampled_docs = random.sample(topic_docs, n)

    print("\n" + "=" * 120)
    print(f"TOPIC {topic_id}")
    print(f"CLUSTER SIZE : {len(topic_docs)}")
    print(f"SAMPLE SIZE  : {n}")
    print("=" * 120)

    for i, doc in enumerate(sampled_docs, 1):
        print(f"{i}. {doc}")


TOPIC 0
CLUSTER SIZE : 384
SAMPLE SIZE  : 20
1. verifikasi wajahnya susah sekali sudah paling terang tmptnya tetap saja tidak ada respon
2. sudah 2 hari verifikasi wajah gagal bae capek deh
3. login tidak bisa untuk proses verifikasi wajah
4. tixak bksa scan wajah gagal terus
5. verifikasi wajah yang terlalu rumit harus berulang2
6. scan wajah gagal terus padahal saya sendiri orangnya
7. aplikasi kok tol verifikasi muka saja sulit
8. buka rekening online ribet di verifikasi wajah selalu gagal apakah wajahku sejelek itu bagimu p
9. enggak bisa verifikasi wajah padahal sudah sangat jelas email enggak di respon
10. sungguh parah verifikasi wajah perintah sudah diikuti selalu gagal sampai jengkel harus wajah ganteng mungkin ya baru ok
11. prepikasi wajah untuk register sangat susah sudah mengikuti petunjuk tetap enggak bisa
12. aplikasi aneh verifikasi wajah enggak berhasil di tolak terus terus ada keterangan pendaftaran mencurigakan kan aneh
13. saya tidak dapat verifikasi wajah
14. veri

NPMI

In [56]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [57]:
doc.split()

['kenapa', 'tidak', 'bisa', 'kirim', 'sms', 'verifikasi', 'pak']

In [58]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [59]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)
topic_words = []

for topic in topic_info.Topic:

    if topic == -1:
        continue

    words = []

    for word, score in topic_model.get_topic(topic):
        if word in dictionary.token2id:
            words.append(word)
    # Need at least 2 words for coherence
    if len(words) >= 2:
        topic_words.append(words)

In [60]:
# sanity check
print(f"Valid Topics : {len(topic_words)}")

print()

print(topic_words[:3])

Valid Topics : 138

[['verifikasi wajah', 'wajah', 'wajah gagal', 'wajah susah', 'wajahnya', 'susah verifikasi', 'verifikasi wajahnya', 'banget verifikasi', 'wajah selalu', 'verifikasi'], ['bni mobile', 'mobile', 'mobile banking', 'bni', 'mending bni', 'banking', 'bagus bni', 'aplikasi bni', 'wonder bni', 'mobile lebih'], ['pemeliharaan', 'maintenance', 'tiap malam', 'malam', 'pemeliharaan sistem', 'pemeliharaan rutin', 'rutin', 'malam maintenance', 'hari pemeliharaan', 'malam pemeliharaan']]


In [61]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : -0.0289


DBCV

In [62]:
mask = np.array(topics) != -1
X = topic_model.umap_model.embedding_[mask].astype(np.float64)
labels = np.array(topics)[mask]

dbcv_score = validity_index(X, labels)
print(f"DBCV : {dbcv_score:.4f}")

DBCV : 0.2203


In [63]:
import pandas as pd
from scipy.stats import chi2_contingency

df["topic"] = topics

# 1. Baseline: proporsi tiap bank di keseluruhan korpus
baseline = df["bank"].value_counts(normalize=True) * 100
print("Proporsi bank di keseluruhan korpus (baseline):")
print(baseline.round(2))
print()

# 2. Proporsi tiap bank DI DALAM tiap topik
crosstab = pd.crosstab(df["topic"], df["bank"], normalize="index") * 100
crosstab = crosstab.round(2)

# 3. Hitung "lift" = proporsi di topik / proporsi baseline
#    >1 artinya over-represented di topik itu, <1 artinya under-represented
lift = crosstab.copy()
for bank in baseline.index:
    lift[bank] = crosstab[bank] / baseline[bank]

# 4. Tandai topik yang "njomplang" (deviasi lift > 1.5x atau < 0.5x dari baseline)
def flag_imbalance(row):
    return any(row > 1.5) or any(row < 0.5)

lift["is_imbalanced"] = lift[baseline.index].apply(flag_imbalance, axis=1)

# gabung count per topik biar gampang liat mana yang topik "besar" (bukan cuma noise kecil)
topic_sizes = df[df["topic"] != -1]["topic"].value_counts()
lift["topic_size"] = lift.index.map(topic_sizes)

result = lift[lift.index != -1].sort_values("is_imbalanced", ascending=False)
print(result[list(baseline.index) + ["is_imbalanced", "topic_size"]])

Proporsi bank di keseluruhan korpus (baseline):
bank
BRIMO_REVIEWS            29.27
WONDR_BNI_REVIEWS        26.60
LIVIN_MANDIRI_REVIEWS    26.05
BCAMOBILE_REVIEWS        18.09
Name: proportion, dtype: float64

bank   BRIMO_REVIEWS  WONDR_BNI_REVIEWS  LIVIN_MANDIRI_REVIEWS  \
topic                                                            
0           0.356021           1.850674               1.209727   
82          0.471164           0.648225               0.264904   
100         3.416707           0.000000               0.000000   
98          0.069701           0.000000               0.078319   
96          0.000000           0.150400               3.608832   
...              ...                ...                    ...   
28          0.921827           0.984745               0.975152   
26          0.756459           1.033249               1.465416   
25          0.872285           0.800129               1.116434   
110         1.166805           1.284041               0.655349 

finding the best settings for both HDBSCAN and UMAP

In [64]:
# import itertools

# param_grid = {
#     "min_cluster_size": [50, 75, 100],
#     "min_samples": [5, 10],
#     "cluster_selection_epsilon": [0.0, 0.05, 0.1],
# }

# results = []
# combos = list(itertools.product(*param_grid.values()))
# print(f"Total kombinasi yang dicoba: {len(combos)}")

# for mcs, ms, eps in combos:
#     hdbscan_test = HDBSCAN(
#         min_cluster_size=mcs,
#         min_samples=ms,
#         metric="euclidean",
#         cluster_selection_method="leaf",
#         cluster_selection_epsilon=eps,
#         prediction_data=True,
#     )
#     tm = BERTopic(
#         embedding_model=None, calculate_probabilities=False,
#         vectorizer_model=vectorizer_model, ctfidf_model=ctfidf_model,
#         umap_model=umap_model, hdbscan_model=hdbscan_test, verbose=False,
#     )
#     tpcs, _ = tm.fit_transform(documents, embeddings)

#     ti = tm.get_topic_info()
#     n_topics = len(ti) - 1
#     outlier_pct = (np.array(tpcs) == -1).sum() / len(tpcs) * 100
#     max_share = ti[ti.Topic != -1]["Count"].max() / len(tpcs) * 100 if n_topics > 0 else 0
#     mask = np.array(tpcs) != -1
#     sil = silhouette_score(tm.umap_model.embedding_[mask], np.array(tpcs)[mask]) if len(set(np.array(tpcs)[mask])) > 1 else float("nan")

#     row = {"min_cluster_size": mcs, "min_samples": ms, "epsilon": eps,
#            "topics": n_topics, "outlier_%": round(outlier_pct, 2),
#            "max_topic_share_%": round(max_share, 2), "silhouette": round(sil, 4)}
#     results.append(row)
#     print(row)

# results_df = pd.DataFrame(results).sort_values("outlier_%")
# results_df